<a href="https://colab.research.google.com/github/maanaav15369/AI/blob/main/pytorch_lstm_next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

In [3]:
document = """Machine Learning is a branch of artificial intelligence that allows computers to learn from data without being explicitly programmed. Machine learning algorithms identify patterns in data and use those patterns to make predictions or decisions.

There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train a model. Unsupervised learning works with unlabeled data and tries to discover hidden patterns. Reinforcement learning allows an agent to learn by interacting with an environment and receiving rewards or penalties.

Python is one of the most popular programming languages for machine learning and data science. Python provides many powerful libraries such as NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn, TensorFlow, and Keras.

NumPy is used for numerical computations and provides powerful tools for working with arrays. Pandas is mainly used for data manipulation and data analysis. Matplotlib and Seaborn are commonly used for data visualization. Scikit-learn provides many machine learning algorithms and utilities for preprocessing and model evaluation.

Deep learning is a subset of machine learning that uses artificial neural networks with multiple layers. Neural networks are inspired by the structure of the human brain. A neural network consists of input layers, hidden layers, and an output layer.

Each neuron receives some inputs, applies weights to those inputs, adds a bias, and passes the result through an activation function. Common activation functions include ReLU, sigmoid, and tanh.

A neural network learns by adjusting its weights during training. The model makes a prediction, calculates the error between the prediction and the actual value, and updates the weights to reduce the error. This process is called backpropagation.

Recurrent Neural Networks, also called RNNs, are designed to work with sequential data. Sequential data includes text, speech, time series, and other data where the order of information matters.

RNNs maintain a hidden state that stores information from previous time steps. This allows an RNN to use information from earlier words when processing the current word. However, traditional RNNs can have difficulty remembering information over long sequences.

Long Short-Term Memory networks, commonly known as LSTMs, are a special type of recurrent neural network. LSTM networks were designed to solve the long-term dependency problem of traditional RNNs.

An LSTM contains three important gates: the forget gate, the input gate, and the output gate. The forget gate decides which information should be removed from the cell state. The input gate decides which new information should be stored. The output gate decides which information should be used as the output.

LSTMs are very useful for natural language processing tasks. They can be used for text generation, sentiment analysis, language modeling, speech recognition, and next word prediction.

Next word prediction is the task of predicting the most likely word that comes after a given sequence of words. For example, if the input sentence is "machine learning is", the model might predict "powerful" as the next word.

To build a next word prediction model, we first need a text dataset. The text is cleaned and converted into tokens. Tokenization means converting words into numerical representations that can be understood by a neural network.

For example, the sentence "I love machine learning" can be converted into tokens such as I, love, machine, and learning. Each unique word is assigned an integer index.

After tokenization, we create sequences from the text. Suppose the sentence is "I love machine learning". We can create training sequences such as "I love", "I love machine", and "I love machine learning".

The input sequence is used by the model to predict the next word. For example, the input "I love machine" can have "learning" as the target word.

Padding is often used because neural networks generally require input sequences to have the same length. Shorter sequences are padded with zeros so that all sequences have equal length.

An embedding layer converts word indices into dense numerical vectors. Words with similar meanings can have similar vector representations. The embedding layer helps the model learn useful relationships between words.

After the embedding layer, an LSTM layer processes the sequence and learns dependencies between words. The output of the LSTM is passed to a Dense layer that produces probabilities for all words in the vocabulary.

The final layer commonly uses the softmax activation function. Softmax converts the output values into probabilities. The word with the highest probability can be selected as the predicted next word.

For example, suppose the input is "I am learning". The model may calculate probabilities for words such as Python, machine, programming, and data. If Python has the highest probability, the model predicts Python as the next word.

The model is trained using many input-output pairs. During training, the model learns which words are likely to appear after particular sequences of words.

The loss function measures how different the predicted word is from the actual next word. Categorical cross entropy or sparse categorical cross entropy is commonly used for next word prediction.

The optimizer updates the weights of the neural network during training. Adam is a popular optimizer because it generally provides good performance and requires relatively little tuning.

The number of epochs determines how many times the model sees the complete training dataset. Increasing the number of epochs can improve learning, but training for too many epochs can cause overfitting.

Overfitting happens when a model learns the training data too closely and performs poorly on new data. Dropout is a regularization technique that can help reduce overfitting.

The quality of a next word prediction model depends heavily on the quality and size of the training text. A larger and more diverse dataset can help the model learn more vocabulary and language patterns.

After training the model, we can provide a starting sentence and ask the model to predict the next word. The predicted word can then be added to the sentence and used as input again to predict another word.

For example, if the starting sentence is "machine learning", the model predicts "is". The new sentence becomes "machine learning is". The model can then predict another word such as "powerful". This process can continue to generate a complete sentence.

Temperature can be used to control the randomness of text generation. A lower temperature generally produces more predictable words, while a higher temperature produces more diverse and random predictions.

A next word prediction model does not truly understand language like a human. Instead, it learns statistical patterns and relationships between words from the training data.

LSTM models can be useful for learning sequential relationships, but modern natural language processing systems often use transformer-based architectures. Transformers can process sequences more efficiently and are widely used in modern language models.

The complete workflow for an LSTM next word predictor is: collect text, clean the text, tokenize the text, create input sequences, pad the sequences, create input and target data, build the LSTM model, train the model, evaluate the model, and generate new text.

The main components of an LSTM next word prediction project are the tokenizer, input sequences, padding, embedding layer, LSTM layer, dense layer, softmax activation, loss function, optimizer, and text generation function.

The goal of the project is to train a neural network that learns the patterns in a text corpus and predicts the most probable next word for a given sequence of words."""

In [4]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
# tokenize
tokens = word_tokenize(document.lower())

In [6]:
# build vocab
vocab = {'<unk>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

vocab

{'<unk>': 0,
 'machine': 1,
 'learning': 2,
 'is': 3,
 'a': 4,
 'branch': 5,
 'of': 6,
 'artificial': 7,
 'intelligence': 8,
 'that': 9,
 'allows': 10,
 'computers': 11,
 'to': 12,
 'learn': 13,
 'from': 14,
 'data': 15,
 'without': 16,
 'being': 17,
 'explicitly': 18,
 'programmed': 19,
 '.': 20,
 'algorithms': 21,
 'identify': 22,
 'patterns': 23,
 'in': 24,
 'and': 25,
 'use': 26,
 'those': 27,
 'make': 28,
 'predictions': 29,
 'or': 30,
 'decisions': 31,
 'there': 32,
 'are': 33,
 'three': 34,
 'main': 35,
 'types': 36,
 ':': 37,
 'supervised': 38,
 ',': 39,
 'unsupervised': 40,
 'reinforcement': 41,
 'uses': 42,
 'labeled': 43,
 'train': 44,
 'model': 45,
 'works': 46,
 'with': 47,
 'unlabeled': 48,
 'tries': 49,
 'discover': 50,
 'hidden': 51,
 'an': 52,
 'agent': 53,
 'by': 54,
 'interacting': 55,
 'environment': 56,
 'receiving': 57,
 'rewards': 58,
 'penalties': 59,
 'python': 60,
 'one': 61,
 'the': 62,
 'most': 63,
 'popular': 64,
 'programming': 65,
 'languages': 66,
 'for'

In [7]:
len(vocab)

397

In [8]:
input_sentences = document.split('\n')

In [9]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [10]:
input_numerical_sentences = []

for sentence in input_sentences:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))


In [11]:
len(input_numerical_sentences)

71

In [12]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [13]:
len(training_sequence)

1368

In [14]:
training_sequence[:5]

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]

In [15]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

60

In [16]:
training_sequence[0]

[1, 2]

In [17]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [18]:
len(padded_training_sequence[10])

60

In [19]:
padded_training_sequence = torch.tensor(padded_training_sequence, dtype=torch.long)

In [20]:
padded_training_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0,   0,  ..., 227, 228,   6],
        [  0,   0,   0,  ..., 228,   6, 172],
        [  0,   0,   0,  ...,   6, 172,  20]])

In [21]:
X = padded_training_sequence[:, :-1]
y = padded_training_sequence[:,-1]

In [22]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0,   0,  ...,   4, 227, 228],
        [  0,   0,   0,  ..., 227, 228,   6],
        [  0,   0,   0,  ..., 228,   6, 172]])

In [23]:
y

tensor([  2,   3,   4,  ...,   6, 172,  20])

In [24]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [25]:
dataset = CustomDataset(X,y)

In [26]:
len(dataset)

1368

In [27]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [28]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.fc = nn.Linear(150, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [29]:
model = LSTMModel(len(vocab))

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
model.to(device)

LSTMModel(
  (embedding): Embedding(397, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=397, bias=True)
)

In [32]:
epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [33]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 246.3251
Epoch: 2, Loss: 214.4054
Epoch: 3, Loss: 198.1210
Epoch: 4, Loss: 181.4055
Epoch: 5, Loss: 164.3979
Epoch: 6, Loss: 148.2590
Epoch: 7, Loss: 132.6659
Epoch: 8, Loss: 118.1940
Epoch: 9, Loss: 104.8641
Epoch: 10, Loss: 92.0484
Epoch: 11, Loss: 80.4321
Epoch: 12, Loss: 70.1007
Epoch: 13, Loss: 60.6208
Epoch: 14, Loss: 52.5093
Epoch: 15, Loss: 45.3369
Epoch: 16, Loss: 39.0303
Epoch: 17, Loss: 33.5772
Epoch: 18, Loss: 29.0369
Epoch: 19, Loss: 25.3629
Epoch: 20, Loss: 22.1792
Epoch: 21, Loss: 19.4796
Epoch: 22, Loss: 17.1712
Epoch: 23, Loss: 15.1501
Epoch: 24, Loss: 13.6626
Epoch: 25, Loss: 12.2478
Epoch: 26, Loss: 11.0793
Epoch: 27, Loss: 10.0154
Epoch: 28, Loss: 9.0700
Epoch: 29, Loss: 8.3236
Epoch: 30, Loss: 7.6915
Epoch: 31, Loss: 7.1593
Epoch: 32, Loss: 6.6635
Epoch: 33, Loss: 6.1709
Epoch: 34, Loss: 5.7991
Epoch: 35, Loss: 5.4458
Epoch: 36, Loss: 5.1320
Epoch: 37, Loss: 4.8600
Epoch: 38, Loss: 4.6217
Epoch: 39, Loss: 4.3370
Epoch: 40, Loss: 4.1194
Epoch: 41, Lo

In [36]:
def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding (max_len - 1 = 60 - 1 = 59)
  padded_text = torch.tensor([0] * (59 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0).to(device)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]

In [38]:
prediction(model, vocab, "The goal of the project is to train a neural network that learns ")

'The goal of the project is to train a neural network that learns  the'

In [42]:
new_input_text = "The goal of the project is to train a neural network that learns the"
next_prediction = prediction(model, vocab, new_input_text)
print(next_prediction)


The goal of the project is to train a neural network that learns the patterns


In [43]:
import time

num_tokens = 10
input_text = "Deep learning is a"

for i in range(num_tokens):
  output_text = prediction(model, vocab, input_text)
  print(output_text)
  input_text = output_text
  time.sleep(0.5)


Deep learning is a subset
Deep learning is a subset of
Deep learning is a subset of machine
Deep learning is a subset of machine learning
Deep learning is a subset of machine learning that
Deep learning is a subset of machine learning that uses
Deep learning is a subset of machine learning that uses artificial
Deep learning is a subset of machine learning that uses artificial neural
Deep learning is a subset of machine learning that uses artificial neural networks
Deep learning is a subset of machine learning that uses artificial neural networks with


In [40]:
dataloader1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [41]:
# Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to compute gradients
        for batch_x, batch_y in dataloader1:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Get model predictions
            outputs = model(batch_x)

            # Get the predicted word indices
            _, predicted = torch.max(outputs, dim=1)

            # Compare with actual labels
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")


Model Accuracy: 98.90%
